# Laboratorio 4. Análisis de Datos GeoEspaciales

Fabian Prado Dluzniewski 23427

Abby Donis 22440

Hansel Lopez 19026

## Contexto

Los lagos de Atitlán y Amatitlán llevan décadas mostrando signos de deterioro, sobre todo por
la proliferación de cianobacterias: microorganismos que forman floraciones potencialmente
tóxicas cuando el agua está caliente, quieta y cargada de nutrientes.

Medir eso con muestreos de campo es caro, lento y cubre pocos puntos. La alternativa es la
observación desde satélite. La misión **Sentinel-2** del programa Copernicus pasa cada pocos
días sobre Guatemala y registra la luz que devuelve el agua en 13 bandas distintas, incluida
una puesta justo donde la clorofila de las cianobacterias deja su firma óptica.

Este laboratorio reconstruye 18 meses de historia de ambos lagos a partir de esas imágenes.

Este notebook funciona como índice. El trabajo está repartido en seis notebooks, uno por
bloque de ejercicios.

## Estructura

| Notebook | Contenido | Ejercicios | Puntos |
|---|---|---|---|
| `01_Descarga_Datos.ipynb` | Conexión al API de Sentinel-2 y obtención de los rasters | 1 y 2 | 15 |
| `02_Indices.ipynb` | Índice de cianobacteria, NDVI y NDWI, con sus mapas | 3 | 20 + 10 |
| `03_Analisis_Temporal.ipynb` | Evolución en el tiempo, picos y fechas críticas | 4 | 20 |
| `04_Analisis_Espacial.ipynb` | Distribución dentro de cada lago, mapas interactivos | 5 | parte de 20 |
| `05_Correlaciones_y_Comparacion.ipynb` | NDVI/NDWI contra cianobacteria; comparación de lagos | 6 y 7 | 20 |
| `06_Analisis_Adicional.ipynb` | Extensión, persistencia, distribuciones y estacionalidad | 8 | 15 |

El orden de lectura es el mismo. Cada notebook deja en disco lo que necesita el siguiente, de
modo que se pueden correr por separado una vez descargadas las imágenes.

La carpeta `avance/` guarda la primera versión del laboratorio, la que se entregó el 13 de
agosto con los ejercicios 1 al 4. Se conserva sin modificar porque documenta esa entrega y
forma parte del historial de contribuciones del grupo. La versión final se rehízo sobre otra
base por tres razones técnicas que están explicadas en `avance/README.md` y desarrolladas en
los cuadernos de esta carpeta.

El código compartido vive en `src/`, para que los seis notebooks trabajen sobre exactamente las
mismas definiciones:

| Módulo | Contenido |
|---|---|
| `src/config.py` | Áreas de interés, las 22 fechas oficiales, bandas y rutas |
| `src/descarga.py` | Descarga de las escenas con openEO |
| `src/indices.py` | Script CyanoLakes traducido a Python, más NDVI y NDWI |
| `src/datos.py` | Lectura de escenas y construcción de las tablas de análisis |
| `src/graficos.py` | Rampa de color oficial, realce y reproyección para folium |

## Origen de los datos

Las imágenes vienen de la colección **`SENTINEL2_L1C`** del Copernicus Data Space Ecosystem,
accedida por programa con el módulo `openeo`.

La elección de L1C sobre L2A no es la obvia y es un resultado del propio laboratorio. El script
de cianobacteria está calibrado para L1C —lo dice su nombre— y al probarlo sobre L2A el índice
se rompe sobre agua muy clara: la corrección atmosférica deja el rojo y el borde rojo
prácticamente en cero sobre Atitlán, el índice divide una diferencia entre esa suma casi nula,
y el resultado se sale del rango teórico (llegaba a valores de −83 a +55 cuando el NDCI solo
puede vivir entre −1 y 1). Con L1C el cociente vuelve a tener sentido. El detalle, con los
números de la comparación, está en `01_Descarga_Datos.ipynb`.

Se usan **exclusivamente las 22 fechas** que fija el enunciado —11 por lago— para que todos los
grupos trabajen sobre la misma base de imágenes.

No se descargan escenas completas. Para cada fecha se le pide al servidor un cubo recortado al
rectángulo del lago, limitado a un solo día y a **10 de las 13 bandas**, cada una justificada
por algún índice del laboratorio:

| Banda | Para qué |
|---|---|
| B04, B05 | Índice de cianobacteria (NDCI) |
| B04, B08 | NDVI |
| B03, B08 | NDWI |
| B04, B07, B8A | Índice de algas flotantes (FAI) |
| B02, B03, B04, B08, B11, B12 | Máscara de cuerpo de agua |
| B10 | Descarte de nube alta (banda de cirrus, exclusiva de L1C) |

Todo se trabaja a **20 metros por píxel**. Las bandas del borde rojo y del infrarrojo de onda
corta, que son las que alimentan el índice de cianobacteria, ya son nativas de 20 m, así que
pedir 10 m no agregaría información real y multiplicaría por cuatro el peso de la descarga.

Los GeoTIFF resultantes **no se versionan**: son varios cientos de MB y se reconstruyen en
cualquier momento con el módulo de descarga. La carpeta `Lab4/data/` está en el `.gitignore`.

## El índice de cianobacteria

Se usa el script oficial **CyanoLakes Chlorophyll-a**, de Jeremy Kravitz y Mark Matthews (2020),
del repositorio de scripts personalizados de Sentinel Hub:

<https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/cyanobacteria_chla_ndci_l1c/>

El script original es JavaScript y corre dentro de Sentinel Hub devolviendo un color. Para este
laboratorio se tradujo a Python de forma vectorizada, **sin cambiar ninguna fórmula ni ningún
umbral**, por dos razones: se necesita el valor numérico del índice para poder promediarlo y
correlacionarlo, y así el cálculo queda reproducible en el repositorio sin depender de una
sesión del navegador.

El script trabaja en cuatro pasos: delimita el cuerpo de agua combinando seis criterios,
separa la vegetación flotante con el índice FAI, estima la clorofila-a con el NDCI y un
polinomio calibrado, y pinta el resultado con una escala de 26 tramos. Los mapas de este
laboratorio reproducen esa misma escala, así que se ven igual que en Copernicus Browser.

El detalle de la traducción está documentado en `src/indices.py` y explicado paso a paso en
`02_Indices.ipynb`.

## Requisitos

El laboratorio trae su propio entorno virtual con Python 3.13.5. Desde la carpeta `Lab4`:

```
python3 -m venv .venv
.venv/bin/python -m pip install -r requirements.txt
```

`requirements.txt` fija solo las dependencias que los notebooks importan de verdad. En Jupyter
o VS Code hay que seleccionar el intérprete `.venv` antes de correr las celdas. El entorno no se
versiona, está en el `.gitignore`.

### Credenciales de Copernicus

El acceso a Sentinel-2 necesita una cuenta del Copernicus Data Space Ecosystem, que se crea
gratis en <https://dataspace.copernicus.eu>. La autenticación es un flujo de código de
dispositivo: la primera vez el programa imprime una dirección web y un código corto; uno abre
esa dirección en el navegador, inicia sesión y confirma el código.

```
.venv/bin/python -m src.descarga --login
```

`openeo` guarda un token de refresco en el computador, así que las corridas siguientes ya no
piden nada. **No hay ninguna contraseña escrita en el repositorio ni en los notebooks.**

### Descarga de las imágenes

```
.venv/bin/python -m src.descarga
```

Son 22 trabajos por lotes contra el servidor de Copernicus. El módulo se salta lo que ya esté
en disco, así que se puede volver a correr sin problema si algo falla a medias. Para ver el
estado sin descargar nada:

```
.venv/bin/python -m src.descarga --estado
```

## Nota sobre el informe

Además de los notebooks se entrega `Informe_Lab4.pdf`, dirigido a un público ambientalista sin
conocimientos de programación: explica qué se midió, qué se encontró y qué significa, sin
código y sin jerga estadística.

Los notebooks contienen el análisis técnico completo, con las explicaciones e interpretaciones
integradas en celdas markdown junto a cada resultado.